# Day 24 — Lecture 8: Regularization (Lasso Regression)

> Source: Apna College Prime AI/ML • Day 24 Lecture 8 • Video Duration: `09:15`
> Core Objective: Regularization ka concept samajhna aur Lasso (L1) penalty kaise unimportant features ke weights ko `0` karke overfitting rokti hai ye dekhna.
> Key Parameters: `lambda` (regularization strength), L1 penalty `sum(|theta_j|)`
> Dataset Used: Synthetic Multi-Feature & Single-Feature Line Data
> Core Concept: Cost function me `lambda * sum(|theta_j|)` add karne se line ka slope kam hota hai aur unimportant features eliminate ho jate hain

## 🛠️ Step 0: Setup & Regularization Methods Overview

Regularization model ko **generalize** aur **simplify** karta hai jab model overfit ho raha ho.

| Method | Type | Cost Penalty | Key Superpower |
|---|---|---|---|
| **Lasso Regression** | `L1` | $\lambda \sum_{j=1}^n |\theta_j|$ | Weak weights ko **exact `0`** karta hai (Feature Selection) |
| **Ridge Regression** | `L2` | $\lambda \sum_{j=1}^n \theta_j^2$ | Weights ko shrink karta hai (near zero) |
| **ElasticNet** | `L1 + L2` | Dono ka mix | Combined regularization |

In [ ]:
# 1. Libraries import karo
import numpy as np
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.metrics import mean_squared_error

# 2. 5 features ka data banao jisme pehle 2 kaam ke hain aur baaki 3 useless noise hain
np.random.seed(42)
X = np.random.randn(100, 5)
y = 4.0 * X[:, 0] - 2.5 * X[:, 1] + np.random.normal(0, 0.5, 100)
print("Data shape:", X.shape, "| True important weights: [4.0, -2.5, 0, 0, 0]")

## 📍 Step 1: Lasso (L1) Cost Function Math

Normal Linear Regression sirf MSE minimize karta hai, jabki Lasso usme **L1 Penalty** jodta hai:
$$J(\theta) = \frac{1}{2m} \sum_{i=1}^{m} (h_\theta(x^{(i)}) - y^{(i)})^2 + \lambda \sum_{j=1}^{n} |\theta_j|$$
- **Important Note:** Intercept $\theta_0$ ko regularize **NAHI** kiya jata (`j = 1` se `n` tak hi sum hota hai).
- **$\lambda$ (Lambda):** Regularization strength control karta hai.

In [ ]:
# 1. Lasso cost function manually compute karne ka helper banao
def lasso_cost(y_true, y_pred, weights, lam):
    m = len(y_true)
    mse_half = (1 / (2 * m)) * np.sum((y_pred - y_true) ** 2)
    l1_penalty = lam * np.sum(np.abs(weights))
    return mse_half, l1_penalty, mse_half + l1_penalty

# 2. High weights vs smaller weights par total cost compare karo
sample_err = np.array([1.0, 2.0, 1.5])
print("High weights cost:", lasso_cost(sample_err, np.zeros(3), np.array([10.0, 8.0]), lam=1.0))
print("Low  weights cost:", lasso_cost(sample_err, np.zeros(3), np.array([4.0, 2.5]), lam=1.0))

## 🎯 Step 2: Feature Selection — Unimportant Weights Become `0`

Dekho kyunki har feature ka $|\theta_j|$ cost badhata hai, model cost minimize karne ke liye unimportant features ka weight **exact `0.0`** set kar deta hai!

In [ ]:
# 1. Plain LinearRegression vs Lasso fit karo
ols = LinearRegression().fit(X, y)
lasso = Lasso(alpha=0.5).fit(X, y)

# 2. Dono models ke feature weights compare karo
print("OLS Weights  :", np.round(ols.coef_, 3))
print("Lasso Weights:", np.round(lasso.coef_, 3))
print("Dekho useless features 2, 3, 4 ka weight Lasso ne 0.0 kar diya!")

## 🏆 Step 3: Geometric Intuition — Slope Decreases Toward X-Axis

Single feature ($h_\theta(x) = \theta_0 + \theta_1 x_1$) par penalty lagane se $\theta_1$ (slope $m$) kam hota hai, aur best-fit line x-axis ke thoda paas aa kar simpler ban jati hai.

In [ ]:
# 1. Single feature data par noisy outlier rakho
X_1d = np.array([[1], [2], [3], [4], [5]])
y_1d = np.array([2, 4, 6, 8, 15])

# 2. OLS vs Lasso slope (theta_1) compare karo
ols_1d = LinearRegression().fit(X_1d, y_1d)
lasso_1d = Lasso(alpha=2.0).fit(X_1d, y_1d)
print(f"OLS Slope (theta_1)  : {ols_1d.coef_[0]:.3f}")
print(f"Lasso Slope (theta_1): {lasso_1d.coef_[0]:.3f} (Less steep, shifted toward x-axis!)")